[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E9%A1%A7%E5%AE%A2%E8%A9%95%E8%AB%96%E8%88%87%E8%8F%9C%E5%96%AE%E6%AA%94%E6%A1%88.ipynb)

# 🧋 茶語時光：用 Python 讀懂顧客評論與菜單檔案

> 🧋 **情境**：你是「茶語時光」總部新來的資料小幫手。店長丟給你一堆檔案：一份顧客評論文字檔、一份菜單 CSV、一份門市資訊 JSON，還有一份從舊電腦匯出、打開全是亂碼的會員名單。今天的任務是：**不用 Excel，用 Python 把這些檔案讀懂、整理好、再存成店長要的檔案。**

| 素材檔案 | 內容 |
| --- | --- |
| `菜單.csv` | 10 個品項的價格與成本 |
| `門市資訊.json` | 5 家門市的營業時間、外送、熱銷品項（巢狀 JSON） |
| `顧客評論.txt` | 3 月份 120 則顧客評論 |
| `舊系統會員_big5.csv` | 舊 POS 系統匯出的會員檔（Big5 編碼，會亂碼！） |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '菜單.csv',
    '門市資訊.json',
    '顧客評論.txt',
    '舊系統會員_big5.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['菜單.csv', '門市資訊.json', '顧客評論.txt', '舊系統會員_big5.csv']


## 任務 1：顧客到底滿不滿意？——讀取文字檔

**老闆的問題**：「3 月的評論有幾則？平均幾顆星？有多少人抱怨『太甜』？把一、二星的負評整理給我。」

**白話說明**：用 `with open()` 打開文字檔，一行一行讀進來；每一行用 `|` 切開，就能拿到日期、門市、星等、評論。

In [2]:
reviews = []                                          # 準備一個空清單，裝每一則評論
with open('顧客評論.txt', 'r', encoding='utf-8') as f:  # 用 UTF-8 打開文字檔
    for line in f:                                    # 一次讀一行
        line = line.strip()                           # 去掉頭尾空白與換行
        if not line or line.startswith('#'):          # 空行或 # 開頭的說明行就跳過
            continue
        date, store, star, text = line.split('|')     # 用 | 切成四個欄位
        reviews.append({'日期': date, '門市': store, '星等': int(star), '評論': text})

print('評論總數：', len(reviews))
print('平均星等：', round(sum(r['星等'] for r in reviews) / len(reviews), 2))
print('提到「甜」的評論：', sum('甜' in r['評論'] for r in reviews), '則')
print('第一則評論：', reviews[0])

評論總數： 120
平均星等： 3.8
提到「甜」的評論： 19 則
第一則評論： {'日期': '2024-03-01', '門市': '台中逢甲店', '星等': 4, '評論': '芋頭鮮奶料超多，會回購'}


In [3]:
# 把一、二星的負評寫成新的文字檔，交給店長
bad = [r for r in reviews if r['星等'] <= 2]           # 篩出星等 ≤ 2 的評論
with open('負評整理.txt', 'w', encoding='utf-8') as f:   # 'w' 寫入模式：建立新檔（已存在會覆蓋）
    f.write(f'3 月負評共 {len(bad)} 則\n')
    for r in bad:
        f.write(f"{r['日期']}  {r['門市']}  {r['星等']}★  {r['評論']}\n")

print(open('負評整理.txt', encoding='utf-8').read()[:300])   # 印出前 300 個字確認內容

3 月負評共 11 則
2024-03-11  台北西門店  2★  點無糖卻做成全糖，太甜
2024-03-06  高雄巨蛋店  2★  外送超時一小時，冰塊都融了
2024-03-07  台北西門店  2★  店員態度很差
2024-03-09  台北西門店  2★  半糖還是太甜了
2024-03-12  台中逢甲店  2★  等了快 30 分鐘才拿到
2024-03-25  高雄巨蛋店  2★  店員態度很差
2024-03-28  台北西門店  2★  飲料灑出來，杯子沒封好
2024-03-07  台北西門店  2★  外送超時一小時，冰塊都融了
2024-03-18  台北西門店


> 💡 **看懂結果**：負評集中在「太甜（甜度做錯）」、「等太久」、「外送超時」、「服務態度」幾類——把負評分類統計，就是店長下週開會的討論重點。
>
> 🤖 **換你問 AI**：「我有一個文字檔，每行格式是『日期|門市|星等|評論』，請用 Python 內建的 open 讀取，統計每家門市的評論數與平均星等，並把 1～2 星的評論另存成文字檔。」

## 任務 2：哪一杯最賺錢？——讀取 CSV 菜單

**老闆的問題**：「每一杯中杯賣出去，扣掉成本可以賺多少？哪一杯最賺？」

**白話說明**：`csv.DictReader` 會把第一列當欄位名稱，每一列變成一個字典。⚠️ CSV 讀進來的數字都是**文字**，要先用 `int()` 轉成數字才能相減。

In [4]:
import csv

menu = []
with open('菜單.csv', 'r', encoding='utf-8') as f:
    for row in csv.DictReader(f):                     # 每一列變成 {'品項': '珍珠奶茶', '中杯價': '55', ...}
        row['毛利'] = int(row['中杯價']) - int(row['成本'])   # 文字轉數字後再相減
        row['毛利率'] = round(row['毛利'] / int(row['中杯價']) * 100, 1)
        menu.append(row)

for m in sorted(menu, key=lambda m: m['毛利'], reverse=True):   # 依毛利由高到低排序
    print(f"{m['品項']:　<6}  售價 {m['中杯價']:>3}  成本 {m['成本']:>3}  毛利 {m['毛利']:>3}  毛利率 {m['毛利率']}%")

黑糖珍珠鮮奶  售價  75  成本  27  毛利  48  毛利率 64.0%
芋頭鮮奶　　  售價  70  成本  26  毛利  44  毛利率 62.9%
芒果冰沙　　  售價  70  成本  29  毛利  41  毛利率 58.6%
熱可可　　　  售價  60  成本  21  毛利  39  毛利率 65.0%
珍珠奶茶　　  售價  55  成本  17  毛利  38  毛利率 69.1%
冬瓜檸檬　　  售價  50  成本  14  毛利  36  毛利率 72.0%
鮮奶綠　　　  售價  55  成本  20  毛利  35  毛利率 63.6%
多多綠　　　  售價  50  成本  15  毛利  35  毛利率 70.0%
檸檬紅茶　　  售價  40  成本  11  毛利  29  毛利率 72.5%
四季春青茶　  售價  35  成本   7  毛利  28  毛利率 80.0%


> 💡 **看懂結果**：毛利「金額」最高的是高單價的鮮奶系列，但毛利「率」最高的是成本很低的純茶——**賺得多**和**賺得有效率**是兩件事，報告時要說清楚用哪一個指標。
>
> 🤖 **換你問 AI**：「這是菜單 CSV 的前三行：（貼上），請用 csv 模組計算每個品項的毛利與毛利率，並依毛利率排序。」

## 任務 3：哪些門市有外送？——讀取巢狀 JSON

**老闆的問題**：「幫我列出每家店的假日營業時間、有沒有外送，以及熱銷品項。」

**白話說明**：JSON 就像一層一層的資料夾。先找到真正的資料在哪一層：`data['result']['stores']`。

In [5]:
import json

with open('門市資訊.json', 'r', encoding='utf-8') as f:
    data = json.load(f)                               # JSON 檔 → Python 的字典與清單

print('最外層有哪些鍵：', list(data.keys()))
stores = data['result']['stores']                     # 真正的門市清單在 result → stores 這一層

for s in stores:
    delivery = '✅ 有外送' if s['外送'] else '❌ 無外送'
    print(f"{s['門市']}｜假日 {s['營業時間']['假日']}｜{delivery}｜熱銷：{'、'.join(s['熱銷品項'])}")

最外層有哪些鍵： ['status', 'updated', 'result']
台北信義店｜假日 10:00-22:00｜✅ 有外送｜熱銷：鮮奶綠、四季春青茶
台北西門店｜假日 10:00-23:00｜✅ 有外送｜熱銷：珍珠奶茶、芒果冰沙
台中逢甲店｜假日 10:00-23:00｜✅ 有外送｜熱銷：黑糖珍珠鮮奶、珍珠奶茶
台南成大店｜假日 10:00-22:00｜❌ 無外送｜熱銷：珍珠奶茶、冬瓜檸檬
高雄巨蛋店｜假日 10:00-22:00｜✅ 有外送｜熱銷：多多綠、芋頭鮮奶


> 🤖 **換你問 AI**：「這是一份 JSON 的前 20 行（貼上），請告訴我門市資料在哪一層，並用 Python 列出每家門市的假日營業時間。」

## 任務 4：舊系統的會員檔打開全是亂碼！——處理編碼

**老闆的問題**：「舊電腦匯出的會員名單打開都是怪字，救得回來嗎？」

**白話說明**：台灣舊系統常用 **Big5（cp950）** 編碼，新系統用 **UTF-8**。用錯「翻譯字典」就會亂碼或直接報錯。我們用 `try / except`：先試 UTF-8，失敗就改用 cp950。

In [6]:
def read_text_any(path):
    """先用 UTF-8 讀，失敗就改用 Big5（cp950）。"""
    for enc in ['utf-8', 'cp950']:
        try:
            with open(path, 'r', encoding=enc) as f:
                return f.read(), enc
        except UnicodeDecodeError:                    # 解碼失敗：這本字典不對，換下一本
            print(f'用 {enc} 讀取失敗，換下一種編碼…')
    raise ValueError('無法判斷編碼')

text, enc = read_text_any('舊系統會員_big5.csv')
print('成功的編碼：', enc)
print(text)

# 轉存成 UTF-8，之後所有程式都用 UTF-8 讀
with open('會員_utf8.csv', 'w', encoding='utf-8', newline='') as f:
    f.write(text)
print('已另存為 會員_utf8.csv')

用 utf-8 讀取失敗，換下一種編碼…
成功的編碼： cp950
會員編號,姓名,常去門市,累積點數
M0001,陳怡君,台北西門店,322
M0002,林家豪,台中逢甲店,61
M0003,黃雅婷,台南成大店,497
M0004,張志明,高雄巨蛋店,580
M0005,李淑芬,台北信義店,257
M0006,王冠宇,台北西門店,255
M0007,吳宜庭,台中逢甲店,623
M0008,劉俊傑,台南成大店,393

已另存為 會員_utf8.csv


> 💡 **看懂結果**：第一次用 UTF-8 讀會失敗，改用 cp950 才成功。轉存成 UTF-8 之後，這份檔案就「修好」了。
>
> 🤖 **換你問 AI**：「我的 CSV 用 UTF-8 打開是亂碼，可能是 Big5 編碼。請寫一段程式自動嘗試 utf-8、utf-8-sig、cp950，成功後轉存成 UTF-8。」

## 任務 5：把整理結果存成 JSON，交給工程師做網站

**老闆的問題**：「官網要顯示每家店的平均評分，工程師說給他 JSON 就好。」

In [7]:
import json

summary = {}
for r in reviews:                                     # 依門市累加星等與評論數
    s = summary.setdefault(r['門市'], {'評論數': 0, '星等總和': 0})
    s['評論數'] += 1
    s['星等總和'] += r['星等']

result = [{'門市': name, '評論數': s['評論數'], '平均星等': round(s['星等總和'] / s['評論數'], 2)}
          for name, s in summary.items()]
result.sort(key=lambda x: x['平均星等'], reverse=True)

with open('門市評分.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)   # ensure_ascii=False：中文正常顯示；indent=2：排版好讀

print(open('門市評分.json', encoding='utf-8').read())

[
  {
    "門市": "台北信義店",
    "評論數": 18,
    "平均星等": 4.33
  },
  {
    "門市": "台南成大店",
    "評論數": 24,
    "平均星等": 4.08
  },
  {
    "門市": "台中逢甲店",
    "評論數": 25,
    "平均星等": 3.72
  },
  {
    "門市": "台北西門店",
    "評論數": 30,
    "平均星等": 3.57
  },
  {
    "門市": "高雄巨蛋店",
    "評論數": 23,
    "平均星等": 3.48
  }
]


> 💡 **看懂結果**：評分最低的是高雄巨蛋店與台北西門店，剛好都是人潮會突然爆量的門市（演唱會散場、觀光客）。人多時的出餐速度與服務品質，值得優先改善。

---

## ✍️ 練習題

1. 從 `reviews` 找出「台南成大店」的所有評論，並計算它的平均星等。
2. 把菜單中「中杯價 ≥ 60 元」的品項，存成新的 `高價品項.csv`（要有標題列）。
3. 試著打開一個不存在的檔案 `四月評論.txt`，用 `try / except FileNotFoundError` 印出「找不到檔案，請確認檔名」。

## 🔑 參考解答（先自己試試看！）

In [8]:
# 練習 1
tainan = [r for r in reviews if r['門市'] == '台南成大店']
print('台南成大店評論數：', len(tainan), '平均星等：', round(sum(r['星等'] for r in tainan) / len(tainan), 2))

# 練習 2
with open('高價品項.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['品項', '中杯價'])
    for m in menu:
        if int(m['中杯價']) >= 60:
            w.writerow([m['品項'], m['中杯價']])
print(open('高價品項.csv', encoding='utf-8').read())

# 練習 3
try:
    with open('四月評論.txt', encoding='utf-8') as f:
        print(f.read())
except FileNotFoundError:
    print('找不到檔案，請確認檔名')

台南成大店評論數： 24 平均星等： 4.08
品項,中杯價
黑糖珍珠鮮奶,75
芒果冰沙,70
熱可可,60
芋頭鮮奶,70

找不到檔案，請確認檔名
